## Extract SKU/ Project level metrics from PNL Excels
- PNL Old - New format transfer
- PNL 24/25 sku launches analysis
- Matt Stenmark Listing Fees Extract

In [4]:
import re
import pandas as pd
import country_converter as coco
import openpyxl
import os

### [IMPT] Insertion into new format excel file

In [43]:
import logging
import os
import sys
import time

import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string

# No Excel process — edits the .xlsm on disk. keep_vba=True preserves macros in the saved copy (VBA does not run).
_bc = dict(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
)
try:
    logging.basicConfig(**_bc, force=True)
except TypeError:
    logging.basicConfig(**_bc)
log = logging.getLogger("openpyxl_cell")

_t0 = time.perf_counter()
_mark = [_t0]


def _step(label: str) -> None:
    now = time.perf_counter()
    dt = now - _mark[0]
    log.info("%s (%.2fs since previous step, %.2fs total)", label, dt, now - _t0)
    _mark[0] = now


# Always reload CSVs so new columns (e.g. launch_year_Q) are picked up after re-extraction.
extractions = pd.read_csv("extractions.csv")
extractions_proj = pd.read_csv("extractions_proj.csv", header=0)

if "launch_year_Q" not in extractions.columns:
    raise KeyError(
        "extractions.csv must include 'launch_year_Q' for Actual Launch Year/Month (cols K, L). "
        f"Columns: {list(extractions.columns)!r}"
    )

SKU_COL = "sku_name.1" if "sku_name.1" in extractions.columns else "sku_name"
if SKU_COL not in extractions.columns:
    raise KeyError(f"{SKU_COL!r} not in columns: {list(extractions.columns)!r}")

unique_skus = list(extractions[SKU_COL].dropna().unique())
log.info("unique %s count=%d", SKU_COL, len(unique_skus))

_scripts_dir = os.path.join(os.getcwd(), "scripts")
if os.path.isdir(_scripts_dir) and _scripts_dir not in sys.path:
    sys.path.insert(0, _scripts_dir)
import importlib
import pnl_lib.insert_template as _insert_template
importlib.reload(_insert_template)
apply_insertions_to_workbook = _insert_template.apply_insertions_to_workbook  # noqa: E402

import pnl_lib.paths as _paths
importlib.reload(_paths)
excel_path = _paths.default_new_format_template_path()
base, ext = os.path.splitext(excel_path)
new_excel_path = base + "_NPL" + ext

log.info("cwd=%s", os.getcwd())
log.info("excel_path=%s exists=%s", excel_path, os.path.isfile(excel_path))
log.info("new_excel_path=%s", new_excel_path)

wb = None
try:
    _step("start")
    _step("load_workbook(keep_vba=True)")
    wb = load_workbook(excel_path, keep_vba=True)
    _step("workbook loaded")

    _step("apply_insertions_to_workbook (sku-block: Home, SP, VOL_Auto, SALES_DED_TD, COGS, COGS_OTHER, SALES_DED_KA, A&P)")
    apply_insertions_to_workbook(wb, extractions, extractions_proj)
    _step("all sheet insertions finished")

    _step(f"saving to {new_excel_path!r}")
    wb.save(new_excel_path)
    _step("save finished")

    log.info("Done OK")
    print("Saved:", new_excel_path)
except Exception:
    log.exception("openpyxl pipeline failed — see traceback above")
    raise
finally:
    _step("cleanup")
    if wb is not None:
        try:
            close = getattr(wb, "close", None)
            if callable(close):
                close()
            _step("workbook closed")
        except Exception:
            log.exception("workbook cleanup failed")


10:53:47 INFO unique sku_name.1 count=2
10:53:47 INFO cwd=/Users/kai/Work/iNova/inova/inova-pnl-extract
10:53:47 INFO excel_path=/Users/kai/Work/iNova/inova/pnl-data/new_format/iNova LOCAL P&L Project BLANK TEMPLATE_Master V1.0.xlsm exists=True
10:53:47 INFO new_excel_path=/Users/kai/Work/iNova/inova/pnl-data/new_format/iNova LOCAL P&L Project BLANK TEMPLATE_Master V1.0_NPL.xlsm
10:53:47 INFO start (0.04s since previous step, 0.04s total)
10:53:47 INFO load_workbook(keep_vba=True) (0.00s since previous step, 0.04s total)
/opt/anaconda3/envs/inova/lib/python3.10/site-packages/openpyxl/reader/excel.py:237: UserWarning: Data Validation extension is not supported and will be removed
  ws_parser.bind_all()
10:54:29 INFO workbook loaded (41.07s since previous step, 41.11s total)
10:54:29 INFO selecting sheet 'Home Tab' (0.00s since previous step, 41.11s total)
10:54:29 INFO sheet ready (0.00s since previous step, 41.11s total)
10:54:29 INFO writing 2 values to column J from row 6 (0.00s sinc

Saved: /Users/kai/Work/iNova/inova/pnl-data/new_format/iNova LOCAL P&L Project BLANK TEMPLATE_Master V1.0_NPL.xlsm


### Extractions (bef 2024) Helper Functions


- extract_project_name : get project name from filepath
- extract_gate_number : get gate number from filepath
- extract_year : get file/ proj year from file directory

- pivot_df : pivots wide to long df (output) based on a hardcoded index

- checker_bef2023 : for a given file, checks sheet exists, exchange rate exists and equal no. of SKU names vs SKU tables
- extract_exceptions_bef2023 : runs through entire directory, outputting a dictionary of files that fail checks with reasons (using checker_bef2023)
- extract_table_persku : outputs a given table as data dictionary (can be modified to give df) given start end cell (top-left bound) and end excel cell (bottom-right bound) -- e.g C18, I35
- extract_sku_perfile : uses extract_table_persku, matching the sku_names to table info outputting a dict
- extract_sku_bef2023 : uses extract_sku_perfile, running through files in the directory to output a dict

In [19]:
def extract_project_name(filepath):
    """
    Extracts the project name from a filename string.
    The project name is defined as the first item between the first pair of hyphens.
    If the extracted project name contains numbers, outputs a warning message.
    Example: "G1 P&L - Clear - CANZ & AMENA - MR - Inno (I-L).xlsm" -> "Clear"
    Now, always extracts from the last component if '/' is present.
    """
    # Always operate on the filename only (last component after '/')
    filename = filepath.split('/')[-1] if '/' in filepath else filepath
    matches = re.findall(r'-\s*([^-\n\r]+?)\s*-', filename) 
    if matches:
        project_name = matches[0].strip() #extracts the first item between the first pair of hyphens
        return project_name
    else:
        return None

def extract_gate_number(filepath):
    """
    Extracts the gate number (e.g., 'G1', 'G2', etc.) from a filename or path.
    Looks for the first segment before a hyphen in the last path component,
    and returns it if it matches the 'G' followed by a number pattern.
    """
    # Get the last part of the path (after the last '/'), or the whole string if no '/'
    filename = os.path.basename(filepath)
    # Extract the first segment before the first hyphen
    match = re.match(r'^\s*([^-\n\r]+?)\s*-', filename)
    if match:
        item = match.group(1).strip()
        # Look for 'G' followed by a number (e.g., G1, G2, etc.)
        g_match = re.search(r'G\d+', item)
        if g_match:
            return g_match.group(0)
        else:
            return None
    else:
        return None

def extract_year(filepath):
    """
    Extracts the year (e.g., 2023, 2024, 2025, etc.) from a filename or path.
    Looks for a 4-digit number starting with '20' in any component of the path.
    Returns the first such year found as a string, or None if not found.
    """
    # Split the path into components using common delimiters
    components = re.split(r'[_\-\s\.\\/]', filepath)
    for comp in components:
        match = re.search(r'20\d{2}', comp)
        if match:
            return match.group(0)
    return None

def pivot_df(df, pivot_col_idx): #double check the pivot point (currently hardcoded)
    """
    Takes in a wide DataFrame (such as master_df) and pivot idx.
    pivot_col_idx refers to the first col where we have non-identifiers (i.e will pivot)
    Pivots the headers such that new columns 'metric' and 'value' are added,
    and additional rows are added to reflect the table.
    """
    # Reset index to ensure row numbers are correct
    df = df.reset_index(drop=True)
    
    id_cols = df.columns[:pivot_col_idx]
    feature_cols = df.columns[pivot_col_idx:]
    
    # Melt the DataFrame to long format
    long_df = pd.melt(
        df,
        id_vars=id_cols,
        value_vars=feature_cols,
        var_name='metric',
        value_name='value'
    )
    
    return long_df

def get_value_if_left_label(df, search_col, expected_label, limit=50):
    """
    Iterates through a DataFrame column (search_col) to find an expected label (string).
    When the expected label is encountered, returns the value to the right (adjacent column, same row).
    Only checks up to 'limit' rows.
    Returns the first such value found, or None if not found.
    Optional: Also prints the value in cell E33 (row 32, column 4, 0-indexed) as a loading check.

    search_col can be:
        - an integer (column index)
        - a string (column name)
        - a single character Excel column letter (e.g., 'A', 'B', ..., 'Z', 'AA', etc.)
    """

    def excel_col_to_idx(col):
        """Convert Excel column letter(s) to 0-based index."""
        col = col.upper()
        idx = 0
        for c in col:
            if not ('A' <= c <= 'Z'):
                raise ValueError(f"Invalid Excel column letter: {col}")
            idx = idx * 26 + (ord(c) - ord('A') + 1)
        return idx - 1

    # loading check for df -- returns cell E33
    try:
        print("Loading check, cell E33:", df.iloc[32, 4])
    except Exception as e:
        print("Could not print cell E33:", e)

    # Determine the column index to use
    col_idx = None
    if isinstance(search_col, int):
        col_idx = search_col
    elif isinstance(search_col, str):
        # Check if it's a single or double letter Excel column (e.g., 'A', 'AA')
        if search_col.isalpha():
            try:
                col_idx = excel_col_to_idx(search_col)
            except Exception:
                # If not a valid Excel column, try as column name
                if search_col in df.columns:
                    col_idx = df.columns.get_loc(search_col)
                else:
                    raise ValueError(f"Column '{search_col}' not found in DataFrame columns.")
        else:
            # Try as column name
            if search_col in df.columns:
                col_idx = df.columns.get_loc(search_col)
            else:
                raise ValueError(f"Column '{search_col}' not found in DataFrame columns.")
    else:
        raise ValueError("search_col must be an int, str (column name), or Excel column letter.")

    # Get the column data up to the limit
    col_data = df.iloc[:limit, col_idx]

    for i, val in enumerate(col_data):
        if val == expected_label:
            right_col = col_idx + 1
            if right_col < df.shape[1]:
                return df.iloc[i, right_col]
            else:
                return None
    # return None if no matches found
    return None

def checker_bef2023(filepath):
    # 1. Check "Sales Forecast(LC)" sheet exists
    try:
        df = pd.read_excel(filepath, sheet_name="Sales Forecast (LC)", header=None)
    except Exception as e:
        return "no Sales Forecast (LC) sheet found"

    # 2. Check "Description" exists in Col D (col 3, 0-indexed)
    col_d = df.iloc[:40, 3]  # Only check first 40 rows
    if not (col_d == "Description").any():
        return "sheet format different"

    # 3. Extract sku_names from Col D, starting after first "Description"
    sku_names = []
    found_description = False
    for i, val in enumerate(col_d):
        if not found_description:
            if val == "Description":
                found_description = True
            continue
        # Stop if value looks like a year (e.g. 2020, 2021, etc.) or after 40 rows
        if re.match(r"^20\d{2}", str(val)):
            break
        if pd.notna(val) and val != 0:
            sku_names.append(val)
        if len(sku_names) > 40:
            break

    # 4. Count valid tables by checking for non-zero columns in "List Price (LC)" rows
    n_tables = 0
    col_c = df.iloc[:, 2]
    for idx, val in enumerate(col_c):
        if isinstance(val, str) and re.match(r"^List Price.*", val):
            # Check next 4 columns to the right for non-zero or non-NaN
            row = df.iloc[idx, 3:7]
            if row.notna().any() and (row != 0).any():
                n_tables += 1
        # Stop if we hit the end marker or exceed 1000 rows
        if isinstance(val, str) and "Revenue & Margin Impact - Cannibalized or Umbrella effect on existing brand" in val:
            break
        if idx > 1000:
            return "1000 rows exceeded"

    # 5. Check for too many or too few SKUs
    if len(sku_names) > 15:
        return "more than 15 skus"
    if len(sku_names) == 0:
        return "no SKUs found"
    if n_tables != len(sku_names):
        return f"unequal sku names and tables, count: sku: {len(sku_names)}, tables: {n_tables}"

    # 6. Check exchange rate value exists (Valuation (AUD), M14)
    sheet_names = ["Valuation (AUD)", "Valuation(AUD)"]
    exch_rate_found = False
    for sheet in sheet_names:
        try:
            exch_rate = helper_exchange_rate_finder(filepath, sheet)
            if exch_rate is not None and exch_rate != "":
                exch_rate_found = True
                break
        except Exception:
            continue
    if not exch_rate_found:
        return "problem with exchange rate extraction"

    return "pass"

def extract_exceptions_bef2023(directory_path):
    excel_files = [
        f for f in os.listdir(directory_path)
        if f.endswith(('.xlsx', '.xls', '.xlsm')) and not f.startswith('~')
    ]
    total_files = len(excel_files)
    print(f"Found {total_files} Excel files to process.")

    failed_files = {}
    for file_name in excel_files:
        file_path = os.path.join(directory_path, file_name)
        result = checker_bef2023(file_path)
        if result != "pass":
            failed_files[file_name] = result

    print(f"Files that did not pass checker(): {len(failed_files)}")
    for fname, reason in failed_files.items():
        print(f"{fname}: {reason}")

    return failed_files

def helper_extract_excel_table(filepath, sheet_name, start_cell, end_cell): #yet to check dict output errors #old name: extract_table_persku_dict
    """
    Extracts a rectangular table from an Excel file given the top-left and bottom-right cell references (e.g., 'C18', 'I35').

    Args:
        filepath (str): Path to the Excel file.
        start_cell (str): Excel-style cell reference for the top-left cell (e.g., 'C18').
        end_cell (str): Excel-style cell reference for the bottom-right cell (e.g., 'I35').

    Returns:
        pd.DataFrame: DataFrame containing the extracted table.
    """

    def excel_cell_to_indices(cell):
        """
        Convert Excel cell reference (e.g., 'C18') to zero-based (row, col) indices.
        """
        match = re.match(r"([A-Za-z]+)([0-9]+)", cell)
        if not match:
            raise ValueError(f"Invalid cell reference: {cell}")
        col_letters, row_number = match.groups()
        # Convert column letters to number (A=0, B=1, ..., Z=25, AA=26, etc.)
        col = 0
        for char in col_letters.upper():
            col = col * 26 + (ord(char) - ord('A') + 1)
        col -= 1  # zero-based
        row = int(row_number) - 1  # zero-based
        return row, col

    startrow, startcol = excel_cell_to_indices(start_cell)
    endrow, endcol = excel_cell_to_indices(end_cell)

    df = pd.read_excel(filepath, header=None, sheet_name=sheet_name)
    
    table = df.iloc[startrow:endrow+1, startcol:endcol+1].copy()
    table.columns = table.iloc[0]
    table = table.drop(table.index[1]).reset_index(drop=True) # formatting (drop 2nd row from table)
    
    data_dict = {}

    table = table[1:].reset_index(drop=True) # Remove the first row (which is likely a subheader or index row)
    row_headers = table.iloc[:, 0]
    for col in table.columns[1:]:
        inner_dict = {}
        for idx, row_header in enumerate(row_headers):
            value = table.iloc[idx, table.columns.get_loc(col)]
            inner_dict[row_header] = value
        data_dict[col] = inner_dict

    return table

def helper_exchange_rate_finder(filepath, sheet_name):
    """Looks across the range H1 to S20 to find "exchange rate", extracts the value to the right of it.
        Tests the extracted value to ensure it is valid, returning the value as a float, else returns None"""

    try:
        df2 = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    except Exception as e:
        print(f"Error: Failed to load sheet '{sheet_name}': {e}")  # loading error
        return None
    
    # extract all values within grid  (H to S , row 1 to row 20) --> find the cell with value matching "exchange rate" after value.lower() applied
    table = helper_extract_excel_table(filepath, sheet_name, "J3", "S17")
    # Find the item that is "Exchange Rate" (case-insensitive, strip spaces)
    exch_val = None
    # Iterate through the DataFrame to find the cell with "exchange rate"
    for row_idx in range(df2.shape[0]):
        for col_idx in range(df2.shape[1]):
            cell_value = df2.iloc[row_idx, col_idx]
            if str(cell_value).strip().lower() == "exchange rate":
                # Return the value in the same row, 1 column after
                if col_idx + 1 < df2.shape[1]:
                    exch_val = df2.iloc[row_idx, col_idx + 1]
                    if exch_val is None or pd.isna(exch_val) or exch_val == 0 or str(exch_val).strip().lower() == "nan":
                        print("exch_rate extracted is 0, nan, NaN, None")
                        return None
                    # check if exch_val can be cast to a float (i.e., not a string that can't be converted)
                    try:
                        exch_val_float = float(exch_val)
                    except (ValueError, TypeError):
                        print("exch_rate extracted is not a valid float")
                        return None
                    exch_val = exch_val_float
                    return exch_val
                break
    
    print("no exchange rate found") # print(f"Exchange Rate value: {exch_val}")
    return exch_val

def extract_table_persku(filepath, sheet_name, start_cell, end_cell, rename_header_list=None): #returns dictionary of values permutations
    """
    Extracts a rectangular table from an Excel file given the top-left and bottom-right cell references (e.g., 'C18', 'I35').
    
    Args:
        filepath (str): Path to the Excel file.
        start_cell (str): Excel-style cell reference for the top-left cell (e.g., 'C18').
        end_cell (str): Excel-style cell reference for the bottom-right cell (e.g., 'I35').
        rename_header_list (list, optional): List of column headers to rename the extracted table's columns.

    Returns:
        dict: Dictionary containing the extracted table as key-value pairs.
    """

    def excel_cell_to_indices(cell):
        """
        Convert Excel cell reference (e.g., 'C18') to zero-based (row, col) indices.
        """
        match = re.match(r"([A-Za-z]+)([0-9]+)", cell)
        if not match:
            raise ValueError(f"Invalid cell reference: {cell}")
        col_letters, row_number = match.groups()
        # Convert column letters to number (A=0, B=1, ..., Z=25, AA=26, etc.)
        col = 0
        for char in col_letters.upper():
            col = col * 26 + (ord(char) - ord('A') + 1)
        col -= 1  # zero-based
        row = int(row_number) - 1  # zero-based
        return row, col

    startrow, startcol = excel_cell_to_indices(start_cell)
    endrow, endcol = excel_cell_to_indices(end_cell)

    df = pd.read_excel(filepath, header=None, sheet_name=sheet_name)
    
    table = df.iloc[startrow:endrow+1, startcol:endcol+1].copy()
    # Set column headers from the first row
    table.columns = table.iloc[0]
    # Optionally rename headers if provided
    if rename_header_list is not None:
        if len(rename_header_list) != len(table.columns):
            raise ValueError(f"Length of rename_header_list ({len(rename_header_list)}) does not match number of columns ({len(table.columns)})")
        table.columns = rename_header_list
    table = table.drop(table.index[1]).reset_index(drop=True) # formatting (drop 2nd row from table)
    table = table[1:].reset_index(drop=True) # Remove the first row (which is likely a subheader or index row)

    # Take the first row as column headers and first column as row headers
    col_headers = list(table.columns)[1:]  # skip the first column (row headers)
    row_headers = table.iloc[:, 0]         # first column as row headers

    result_dict = {}
    for i, row_header in enumerate(row_headers):
        for col_header in col_headers:
            value = table.iloc[i, table.columns.get_loc(col_header)]
            key = f"{row_header}_{col_header}"
            result_dict[key] = value
    
    return result_dict

def helper_exchange_rate_applier(input_dict, exch_rate):
    """
    For all keys in input_dict that are local currency (LC) values, multiplies them by the LC:AUD conversion rate.
    """

    lc_variables = [
        "Unit Cost (LC)",
        "List Price (LC)",
        "Gross Sales (LC,000)",
        "Trade Spend Per (disc/bonus gds)",
        "Trade Spend (LC,000)",
        "Listing Fees (LC,000)",
        "Avg Net Selling Price (LC)",
        "Pipeline fill (LC,000)",
        "Net Sales (LC,000)", 
        "Gross Margin (LC,000)",
    ]

    # Prepare regex patterns for matching keys
    lc_patterns = [re.compile(rf"^{re.escape(var)}") for var in lc_variables]

    for key, value in list(input_dict.items()):
        if any(pattern.search(key) for pattern in lc_patterns):
            try:
                new_key = key + "_AUDconverted"
                # print(f"Type of item for key '{metric}': {type(value)}")
                # Only multiply if item is a number (int or float)
                if isinstance(value, (int, float)):
                    input_dict[new_key] = value / exch_rate
                    # print("item is int/float, successfully converted, rehashed")
                else:
                    # Try to convert to float if possible
                    input_dict[new_key] = float(value) / exch_rate
                    # print("item is str, successfully converted, rehashed")
            except (ValueError, TypeError):
                # If conversion fails, leave as is
                print(f"conversion failed, unable to convert for {key} with {value}")
                pass
    
    return input_dict

def extract_sku_perfile(filepath, sheet_name="Sales Forecast (LC)"):
    """
    Extracts SKU names and their corresponding tables from the given Excel file.
    Args:
        filepath (str): Path to the Excel file.
        sheet_name (str): Name of the sheet to process.

    Returns:
        dict: {sku_name: table_dict, ...}
    """

    try:
        df = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    except Exception as e:
        return {"error": f"Failed to load sheet '{sheet_name}': {e}"}

    # 0. Try to load "Valuation (AUD)" or "Valuation(AUD)" sheet
    for sheet in ["Valuation (AUD)", "Valuation(AUD)"]:
        try:
            df2 = pd.read_excel(filepath, sheet_name=sheet, header=None)
            sheet_name_exch_rate = sheet
            break
        except Exception as e:
            last_exception = e
    else:
        return {"error": f"Failed to load sheet 'Valuation (AUD)' or 'Valuation(AUD)': {last_exception}"}

    # 0a. Get exchange rate from Valuation (AUD) sheet
    # Extract exchange rate by iterating through row 5 until "Exchange rate" is found
    exch_val = helper_exchange_rate_finder(filepath, sheet_name_exch_rate)

    # 1. Find the first occurrence of "Description" in col D (index 3)
    col_d = df.iloc[:, 3]
    description_idx = None
    for i, val in enumerate(col_d):
        if str(val).strip() == "Description":
            description_idx = i
            break
    if description_idx is None:
        return {"error": "'Description' not found in column D"}

    # 1a. Find the length of a table based on col C (index 2)
    col_c = df.iloc[:, 2]
    table_len = 0
    for i, val in enumerate(col_c):
        if str(val).strip() == "Growth vs PY":
            # Keep iterating through subsequent rows, maintaining a counter until "Gross Margin %" is encountered
            # Return the number of rows in between "Growth vs PY" and "Gross Margin %"
            table_start_idx = i
            counter = 1
            for j in range(i + 1, len(col_c)):
                if str(col_c.iloc[j]).strip() == "Gross Margin %":
                    counter += 1
                    break
                counter += 1
            table_len = counter
            break  # Stop after finding the first table_len
    # print(f"table length: {table_len}")

    # 2. Extract SKU names by concatenating D,E,F,G for each row after "Description"
    sku_names = []
    max_rows = 40
    for i in range(description_idx + 1, min(description_idx + 1 + max_rows, len(df))):
        row = df.iloc[i, 3:7]  # D,E,F,G (columns 3,4,5,6)
        # Stop if first cell looks like a year (e.g. 2020, 2021, etc.)
        if re.match(r"^20\d{2}", str(row.iloc[0])):
            break
        # Only add if at least one cell is not nan/empty
        if row.notna().any():
            sku_name = " ".join([str(x).strip() for x in row if pd.notna(x) and str(x).strip() != ""])
            if sku_name:
                sku_names.append(sku_name)
        if len(sku_names) >= max_rows:
            break

    # 3. For each SKU, find the first occurrence of "20xx" in col D after "Description"
    output_dict = {}
    df_counter = 0
    sku_pointer = 0
    search_start = description_idx + 1
    rename_header_list = ["sku_desc", "launchquarter", "y1", "y2", "y3", "y4", "y5"] #replaces specific 20xx years

    for i in range(search_start, len(df)): 
        val = df.iloc[i, 3]
        if re.match(r"^Q\d$", str(val).strip()):
            print(sku_name)
            row_start = i - 1
            cell_start = f"C{row_start+1}"  # Excel is 1-based
            row_end = row_start + table_len + 2 # +2 accounts for the Header col + Q1/FY/FY/FY/FY/FY row
            cell_end = f"I{row_end}"
            try:
                extract_dict = extract_table_persku(filepath, sheet_name, cell_start, cell_end, rename_header_list)
                extract_dict["LC_to_AUD_exchrate"] = exch_val

                extract_dict_exch = helper_exchange_rate_applier(extract_dict, exch_val) # helper_exchange_rate_applier should be working -- but also appended exchange rate directly instead for excel tabulation
                # print(f"this is the extracted dict: {extract_dict_exch}")
            except Exception as e:
                extract_dict_exch = {"error": f"Failed to extract table: {e}"} #exception gets triggered
            output_dict[sku_names[sku_pointer]] = extract_dict_exch
            sku_pointer += 1
            df_counter += 1
        if df_counter >= len(sku_names):
            break

    return output_dict

def extract_sku_bef2023(directory, show_progress=False):
    # Load sales forecast sheet for each file in directory
    excel_files = [
        f for f in os.listdir(directory)
        if f.endswith(('.xlsx', '.xls', '.xlsm')) and not f.startswith('~')
    ]
    results = []
    year = extract_year(directory)
    total_files = len(excel_files)
    failed_files = {}

    for idx, file_name in enumerate(excel_files, 1):
        if show_progress:
            print(f"[{idx}/{total_files}] Processing: {file_name}")
        file_path = os.path.join(directory, file_name)
        # Run checker() first to flag exceptions
        check_result = checker_bef2023(file_path)
        if check_result != "pass":
            failed_files[file_name] = check_result

        extractions = extract_sku_perfile(file_path, "Sales Forecast (LC)") #is a dictionary with sku as key, value as 

        # to each extracted dictionary, function to modify the values (multiply by the exchange rate)
    
        for sku, indiv_sku_details in extractions.items():
            results.append({
                "filename": file_name,
                "file_year": year,
                "project_name": extract_project_name(file_name),
                "gate_number": extract_gate_number(file_name),
                "sku_name": sku,
                **indiv_sku_details # Use dict unpacking to merge the metrics directly
            })

    # Convert results (list of dicts) to DataFrame
    results_df = pd.DataFrame(results)
    return results_df, failed_files

In [22]:
def extract_standard_gross_margin(directory, output_csv="Standard_Gross_Margin_Extract.csv"):
    """
    For each Excel file in `directory`, read sheet 'Valuation (AUD)' (or 'Valuation(AUD)'):
    - Standard Gross Margin: row 49, H–M → launchQ, y1–y5.
    - Contribution after Marketing (CAM), label row E81: values row 81, H–M → launchQ, y1–y5.

    Writes one CSV row per file (plus filename / project_name / gate_number) and returns the DataFrame.
    """
    period_columns = ["H", "I", "J", "K", "L", "M"]
    sgm_labels = [
        "Standard_Gross_Margin_launchQ",
        "Standard_Gross_Margin_y1",
        "Standard_Gross_Margin_y2",
        "Standard_Gross_Margin_y3",
        "Standard_Gross_Margin_y4",
        "Standard_Gross_Margin_y5",
    ]
    cam_labels = [
        "CAM_launchQ",
        "CAM_y1",
        "CAM_y2",
        "CAM_y3",
        "CAM_y4",
        "CAM_y5",
    ]

    def _read_period_row(ws, row_idx):
        return [ws[f"{col}{row_idx}"].value for col in period_columns]

    results = []
    excel_files = [
        f
        for f in os.listdir(directory)
        if f.endswith((".xlsx", ".xlsm", ".xls")) and not f.startswith("~")
    ]
    for filename in excel_files:
        file_path = os.path.join(directory, filename)
        try:
            wb = openpyxl.load_workbook(file_path, data_only=True, read_only=True)
            sheet_name = None
            for candidate in ("Valuation (AUD)", "Valuation(AUD)"):
                if candidate in wb.sheetnames:
                    sheet_name = candidate
                    break
            if sheet_name is None:
                raise Exception(f"No 'Valuation (AUD)' sheet in {filename}")
            ws = wb[sheet_name]

            margin_values = _read_period_row(ws, 49)
            cam_values = _read_period_row(ws, 81)

            result = {
                "filename": filename,
                "project_name": extract_project_name(filename),
                "gate_number": extract_gate_number(filename),
            }
            for label, value in zip(sgm_labels, margin_values):
                result[label] = value
            for label, value in zip(cam_labels, cam_values):
                result[label] = value
            results.append(result)
            wb.close()
        except Exception as ex:
            print(f"Failed to process {filename}: {ex}")

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    return df


In [25]:
results_df = extract_standard_gross_margin("../pnl-data/PNL_24_Alethea")
results_df.to_csv('test34.csv')

In [28]:
import os
import pandas as pd
import openpyxl
import re

def extract_project_name(filepath):
    """
    Extracts the project name from a filename string.
    The project name is defined as the first item between the first pair of hyphens.
    Now, always extracts from the last component if '/' is present.
    """
    filename = filepath.split('/')[-1] if '/' in filepath else filepath
    matches = re.findall(r'-\s*([^-\n\r]+?)\s*-', filename)
    if matches:
        project_name = matches[0].strip()
        return project_name
    else:
        return None

def extract_gate_number(filepath):
    """
    Extracts the gate number (e.g., 'G1', 'G2', etc.) from a filename or path.
    Looks for the first segment before a hyphen in the last path component,
    and returns it if it matches the 'G' followed by a number pattern.
    """
    filename = os.path.basename(filepath)
    match = re.match(r'^\s*([^-\n\r]+?)\s*-', filename)
    if match:
        item = match.group(1).strip()
        g_match = re.search(r'G\d+', item)
        if g_match:
            return g_match.group(0)
        else:
            return None
    else:
        return None

def extract_standard_gross_margin(directory, output_csv="Standard_Gross_Margin_Extract.csv"):
    """
    Iterates through all Excel files in a directory and extracts Standard Gross Margin values
    from 'Valuation (AUD)' sheet, row 49, columns H–M (launchQ, y1–y5).
    Returns a DataFrame and also writes to a CSV.
    """
    # Column mapping in Excel (1-indexed): H=8, I=9, ..., M=13
    margin_columns = ['H', 'I', 'J', 'K', 'L', 'M']
    margin_labels = [
        'Standard_Gross_Margin_launchQ',
        'Standard_Gross_Margin_y1',
        'Standard_Gross_Margin_y2',
        'Standard_Gross_Margin_y3',
        'Standard_Gross_Margin_y4',
        'Standard_Gross_Margin_y5',
    ]
    results = []
    excel_files = [
        f for f in os.listdir(directory)
        if f.endswith(('.xlsx', '.xlsm', '.xls')) and not f.startswith('~')
    ]
    for filename in excel_files:
        file_path = os.path.join(directory, filename)
        try:
            wb = openpyxl.load_workbook(file_path, data_only=True, read_only=True)
            if "Valuation (AUD)" not in wb.sheetnames:
                raise Exception(f"No 'Valuation (AUD)' sheet in {filename}")
            ws = wb["Valuation (AUD)"]
            row_idx = 49
            margin_values = []
            for col_letter in margin_columns:
                value = ws[f"{col_letter}{row_idx}"].value
                margin_values.append(value)
            result = {
                "filename": filename,
                "project_name": extract_project_name(filename),
                "gate_number": extract_gate_number(filename),
            }
            for label, value in zip(margin_labels, margin_values):
                result[label] = value
            results.append(result)
            wb.close()
        except Exception as ex:
            print(f"Failed to process {filename}: {ex}")
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    return df

# Example usage:
df = extract_standard_gross_margin("../pnl-data/old_format/")
print(df.head())
df.to_csv('test35.csv')
 

                                            filename project_name gate_number  \
0  ANZ Yogi - iGate 3 Scope Change_Kai Shiun Liew...         None        None   

   Standard_Gross_Margin_launchQ  Standard_Gross_Margin_y1  \
0                              0                         0   

   Standard_Gross_Margin_y2  Standard_Gross_Margin_y3  \
0                         0                         0   

   Standard_Gross_Margin_y4  Standard_Gross_Margin_y5  
0                         0                         0  


In [ ]:
# Yearly checkpointed extraction with reload (avoids re-running all years if interrupted)

import os

# Define all year directories in a list for easier scaling/maintenance
year_dirs = [
    ("../pnl-data/2020_flat", "2020"),
    ("../pnl-data/2021_flat", "2021"),
    ("../pnl-data/2022_flat", "2022"),
    ("../pnl-data/2023_flat", "2023"),
]

all_results = []
all_failed = {}

for year_dir, year in year_dirs:
    checkpoint_path = f"extract_checkpoint_{year}.csv"
    failed_path = f"failed_files_{year}.pkl"

    # If checkpoint exists, load it, else run extraction
    if os.path.exists(checkpoint_path):
        print(f"Checkpoint found for {year}. Loading from {checkpoint_path}")
        results_df = pd.read_csv(checkpoint_path)
        # Try loading failed_files log as well, best effort (not critical)
        try:
            import pickle
            if os.path.exists(failed_path):
                with open(failed_path, "rb") as f:
                    failed_files = pickle.load(f)
            else:
                failed_files = {}
        except Exception:
            failed_files = {}
    else:
        print(f"Extracting for {year_dir} ({year}) from scratch")
        results_df, failed_files = extract_sku_bef2023(year_dir)
        # Save checkpoint
        results_df.to_csv(checkpoint_path, index=False)
        # Save failed file log
        try:
            import pickle
            with open(failed_path, "wb") as f:
                pickle.dump(failed_files, f)
        except Exception as ex:
            print("Warning: failed to pickle failed_files log:", ex)
    all_results.append(results_df)
    all_failed.update(failed_files)

# Concatenate all results into one DataFrame in a single step
df_combined = pd.concat(all_results, ignore_index=True)

# Drop unwanted columns robustly if present
for col in ['Unnamed: 0']:
    if col in df_combined.columns:
        df_combined = df_combined.drop(columns=[col])

# Output and save
print(df_combined.head())
df_combined.to_csv("pnlbef2024_volume_extracts.csv", index=False)


# results_df.to_csv("test2.csv")

### Convert to Long Format DF

In [ ]:
reduced_df = pd.read_csv("extracts_combined_24_25_III_sf.csv", header=0)
output = pivot_df(reduced_df, 6)
output.to_csv("snowflake_24_25_forecast_netsales_y1_y3.csv")
output

,filename,sheet_name,file_year,project_name,gate_number,sku_name,metric,value
0,G4 P&L - Rusty - AU NZ - CANZ - Exp (M).xlsm,Project Summary,2024,Rusty,G4,Betadine Ready to Use Gargle 200mL_Australia,Market,Australia
1,G4 P&L - Rusty - AU NZ - CANZ - Exp (M).xlsm,Project Summary,2024,Rusty,G4,Betadine Ready to Use Gargle 60mL_Australia,Market,Australia
2,G4 P&L - Rusty - AU NZ - CANZ - Exp (M).xlsm,Project Summary,2024,Rusty,G4,Betadine Antiseptic Solution 500mL_Australia,Market,Australia
3,G4 P&L - Rusty - AU NZ - CANZ - Exp (M).xlsm,Project Summary,2024,Rusty,G4,Betadine Antiseptic First Aid Cream 45g_Australia,Market,Australia
4,G4 P&L - Rusty - AU NZ - CANZ - Exp (M).xlsm,Project Summary,2024,Rusty,G4,Betadine Ready to Use Gargle 200mL_NZ,Market,NZ
...,...,...,...,...,...,...,...,...
5067,G3 P&L - Skippy - NZ - CANZ - Exp (M).xlsm,Project Summary,2025,Skippy,G3,Sea-Legs Double Strength_NZ,forecast_net_sales_y3,1052190.0
5068,G1 P&L-Ferrous Wheel-SA-RSA-Inno (R&D).xlsm,Project Summary,2025,Ferrous Wheel,G1,FERROUS FORTE SOMAL SACHET 30_RSA,forecast_net_sales_y3,545257.801505
5069,G1 P&L-Ferrous Wheel-SA-RSA-Inno (R&D).xlsm,Project Summary,2025,Ferrous Wheel,G1,FERROUS FORTE SOMAL DROPS 30 ML_RSA,forecast_net_sales_y3,95064.483091
5070,G1 P&L-Ferrous Wheel-SA-RSA-Inno (R&D).xlsm,Project Summary,2025,Ferrous Wheel,G1,FERROUS FORTE SOMAL SYRUP 150 ML_RSA,forecast_net_sales_y3,331379.61868


### Adhoc column addition (Exchange Rate)
- Extract exch_value with key = file_name
- Map the combined extracts 20-23 with a join to the filename

In [3]:
import os
import pandas as pd

def extract_exchange_rates_from_directory(directory):
    """
    Recursively scans the given directory (including all subdirectories) for Excel files,
    extracts the exchange rate from each file using helper_exchange_rate_finder, and returns
    a DataFrame with columns 'filename' (relative path from root) and 'exch_val'.
    Also returns an error_log dictionary for files where extraction failed.
    Includes a progress tracker: prints completed files over total files.
    """
    import sys

    results = []
    error_log = {}

    # First, collect all Excel files to get the total count
    excel_files = []
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith(('.xlsx', '.xls', '.xlsm')) and not filename.startswith('~'):
                filepath = os.path.join(root, filename)
                rel_path = os.path.relpath(filepath, directory)
                excel_files.append((filepath, rel_path))

    total_files = len(excel_files)
    completed = 0

    for filepath, rel_path in excel_files:
        try:
            exch_val = helper_exchange_rate_finder(filepath, sheet_name="Valuation (AUD)")
            # If not found, try "Valuation(AUD)" as a fallback
            if exch_val is None:
                exch_val = helper_exchange_rate_finder(filepath, sheet_name="Valuation(AUD)")
            if exch_val is None:
                error_log[rel_path] = "Exchange rate not found or invalid"
            results.append({"filename": rel_path, "exch_val": exch_val})
        except Exception as e:
            error_log[rel_path] = f"Error: {e}"
            results.append({"filename": rel_path, "exch_val": None})
        completed += 1
        print(f"Progress: {completed}/{total_files} files processed", end='\r')
        sys.stdout.flush()

    print(f"\nDone. Processed {completed} files.")
    df = pd.DataFrame(results)
    return df, error_log

df, error_log = extract_exchange_rates_from_directory("2020_2023_flat")


Done. Processed 0 files.


### [OLD] Run extractions for 2024/2025 PNL

In [ ]:

def extract_all_proj_summary_aft2024(directory_path):
    """
    Iterates through all files in the given directory, runs extract_project_summary() on each,
    and appends the resulting DataFrames to a master DataFrame as additional rows.
    Returns the concatenated master DataFrame.
    Includes a progress counter of all items.
    Also tracks and returns a list of filenames that were not processed (due to errors or empty DataFrames),
    with the reason for failure: 'no sheet found', 'more than 10 skus', 'check irregular row arrangement - index out of bounds', or 'others'.
    """
    print(f"Starting extract_all_project_summaries. Current working directory: {os.getcwd()}")
    master_df = None
    not_processed_files = {}

    excel_files = [
        f for f in os.listdir(directory_path)
        if f.endswith(('.xlsx', '.xls', '.xlsm')) and not f.startswith('~')
    ]
    
    total_files = len(excel_files)
    print(f"Found {total_files} Excel files to process.")

    for idx, filename in enumerate(excel_files, 1):
        file_path = os.path.join(directory_path, filename)
        print(f"[{idx}/{total_files}] Processing: {file_path}")

        try:
            df, fail_reason = extract_project_summary(file_path, return_fail_reason=True)
            print(f"printing df: {df}")
            # If df is None or empty, consider as not processed
            if df is not None and not df.empty:
                if master_df is None:
                    master_df = df
                else:
                    master_df = pd.concat([master_df, df], ignore_index=True)
            else:
                not_processed_files[filename] = fail_reason
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            # Try to detect if it's an index error
            if "index" in str(e).lower() and "out of bounds" in str(e).lower():
                reason = "check irregular row arrangement - index out of bounds"
            else:
                reason = "others"
            not_processed_files[filename] = reason
            continue

    print(f"Finished processing {total_files} files.")
    if not_processed_files:
        print(f"The following files were not processed (due to errors or empty data):")
        for filename, reason in not_processed_files.items():
            print(f" - {filename}: {reason}")
    else:
        print("All files processed successfully.")
    # Optionally, return both master_df and not_processed_files
    return master_df, not_processed_files

def extract_project_summary(filepath, return_fail_reason=False):
    # Load the Excel with correct headers and check if loaded successfully
    try:
        df = pd.read_excel(
            f"{filepath}", #change your directory for file loading
            sheet_name="Project Summary",
            header=None
        )
        # print("Excel file loaded successfully. Shape:", df.shape)
    except Exception as e:
        print("Error loading Excel file:", e)
        if return_fail_reason:
            return None, "no sheet found"
        else:
            return None

    # extract 3 identifiers
    gate_no = extract_gate_number(filepath)
    proj_name = extract_project_name(filepath)

    # Identify product names and regions
    # product_names = df.iloc[4:14, 2]
    # Dynamically extract product names from col 2 until "GTN % excluding Launch yr one-time costs" is encountered
    product_names = []
    start_row = 4
    col_idx = 2
    fail_reason = None
    try:
        while True and start_row < 30: #hard coded stop limit at 30
            val = df.iloc[start_row, col_idx]
            if isinstance(val, str) and "GTN % excluding Launch yr one-time costs" in val:
                break
            product_names.append(val)
            start_row += 1
        product_names = pd.Series(product_names)
    except Exception as e:
        print(f"Index error or irregular row arrangement in {filepath}: {e}")
        if return_fail_reason:
            return None, "check irregular row arrangement - index out of bounds"
        else:
            return None

    # Filter out values that are just '0', contain 'input' (case-insensitive), are NaN, or are the string 'nan'
    product_names = product_names[~product_names.astype(str).str.fullmatch(r'0', na=False)]
    product_names = product_names[~product_names.astype(str).str.contains(r'input', case=False, na=False)]
    product_names = product_names[product_names.notna()]
    product_names = product_names[~product_names.astype(str).str.lower().eq('nan')]
    # print(product_names)
    
    regions = df.iloc[2, 3:34]
    regions = regions[~regions.astype(str).str.contains(r'total', case=False, na=False)]
    print(regions)

    if len(product_names) > 10:
        print(f"Unable to extract data from {filepath}, more than 10 unique skus")
        if return_fail_reason:
            return pd.DataFrame(), "more than 10 skus"
        else:
            return pd.DataFrame()

    try:
        # Extract values for List Price and GTN
        list_price = df.iloc[4:14, 3:34].values
        gtn_price = df.iloc[17:27, 3:34].values
        launch_onetime_costs = df.iloc[31:41, 3:34].values
        y1_launch_ASP_excl_costs = df.iloc[44:55, 3:34].values
        y2_ongoing_ASP = df.iloc[57:67, 3:34].values
        cogs_per_unit = df.iloc[70:80, 3:34].values
        royalty_per_unit = df.iloc[83:93, 3:34].values
        one_time_costs = df.iloc[96:106, 3:34].values
        gp_perc_y1 = df.iloc[109:119, 3:34].values
        gp_perc_y2_onwards = df.iloc[122:132, 3:34].values

        pipefill_vol = df.iloc[170:180, 3:34].values
        launchyr_vol = df.iloc[185:195, 3:34].values
        y1_total_vol = df.iloc[200:210, 3:34].values
        y2_total_vol = df.iloc[215:225, 3:34].values
        y3_total_vol = df.iloc[230:240, 3:34].values
        y4_total_vol = df.iloc[245:255, 3:34].values
        y5_total_vol = df.iloc[260:270, 3:34].values
    except Exception as e:
        print(f"Index error or irregular row arrangement in {filepath}: {e}")
        if return_fail_reason:
            return None, "check irregular row arrangement - index out of bounds"
        else:
            return None

    # Flatten data
    records = []
    for i, product in enumerate(product_names):
        for j, region in enumerate(regions):
            records.append({
                'sku_name': f"{product}_{region}",
                'country_region': region,
                'list_price': list_price[i][j],
                'gtn_pct_excl_onetime': gtn_price[i][j],
                'launchyr_onetime_cost': launch_onetime_costs[i][j],
                'launchyr_asp_excl_onetime': y1_launch_ASP_excl_costs[i][j],
                'yr2_ongoing_asp_excl_onetime': y2_ongoing_ASP[i][j],
                'cogs_per_unit': cogs_per_unit[i][j],
                'other_cogs_per_unit': royalty_per_unit[i][j],
                'onetime_costs_clearance_woff': one_time_costs[i][j],
                'gp_pct_incl_onetime_launchyr': gp_perc_y1[i][j],
                'gp_pct_yr2_onwards': gp_perc_y2_onwards[i][j],
                'pipefill_volume': pipefill_vol[i][j],
                'launchyr_volume': launchyr_vol[i][j],
                'y1_total_volume': y1_total_vol[i][j],
                'y2_total_volume': y2_total_vol[i][j],
                'y3_total_volume': y3_total_vol[i][j],
                'y4_total_volume': y4_total_vol[i][j],
                'y5_total_volume': y5_total_vol[i][j],
            })

    # Create final dataframe
    result_df = pd.DataFrame(records)

    # Optional: drop rows with missing data if needed
    result_df.dropna(subset=['list_price'], inplace=True)

    # Add new columns at the front of result_df with headers proj_name, gate_no, filename
    result_df.insert(0, 'proj_name', proj_name if proj_name is not None else 'Unknown')
    result_df.insert(1, 'gate_no', gate_no if gate_no is not None else 'Unknown')
    filename = filepath.split('/')[-1]
    result_df.insert(2, 'filename', filename)
    result_df.insert(3, 'year', extract_year(filepath))
    result_df.insert(4, 'sheet_name', 'Project Summary') #MANUALLY INSERTED

    # result_df.insert(1, 'gate_no', gate_no) # table name

    if return_fail_reason:
        return result_df, None
    else:
        return result_df

def get_cell_id_given_name(df, search_col, expected_label, limit=50):
    """
    Iterates through a DataFrame column (search_col) to find an expected label (string).
    When the expected label is encountered, returns the Excel cell ID (e.g., 'A35') for that cell.
    Only checks up to 'limit' rows.
    Returns the first such cell ID found, or None if not found.
    Optional: Also prints the value in cell E33 (row 32, column 4, 0-indexed) as a loading check.

    search_col can be:
        - an integer (column index)
        - a string (column name)
        - a single character Excel column letter (e.g., 'A', 'B', ..., 'Z', 'AA', etc.)
    """

    def excel_col_to_idx(col):
        """Convert Excel column letter(s) to 0-based index."""
        col = col.upper()
        idx = 0
        for c in col:
            if not ('A' <= c <= 'Z'):
                raise ValueError(f"Invalid Excel column letter: {col}")
            idx = idx * 26 + (ord(c) - ord('A') + 1)
        return idx - 1

    def idx_to_excel_col(idx):
        """Convert 0-based column index to Excel column letter(s)."""
        letters = ""
        idx += 1  # 1-based for Excel
        while idx > 0:
            idx, remainder = divmod(idx - 1, 26)
            letters = chr(65 + remainder) + letters
        return letters

    # loading check for df -- returns cell E33
    try:
        print("Loading check, cell E33:", df.iloc[32, 4])
    except Exception as e:
        print("Could not print cell E33:", e)

    # Determine the column index to use
    col_idx = None
    if isinstance(search_col, int):
        col_idx = search_col
    elif isinstance(search_col, str):
        # Check if it's a single or double letter Excel column (e.g., 'A', 'AA')
        if search_col.isalpha():
            try:
                col_idx = excel_col_to_idx(search_col)
            except Exception:
                # If not a valid Excel column, try as column name
                if search_col in df.columns:
                    col_idx = df.columns.get_loc(search_col)
                else:
                    raise ValueError(f"Column '{search_col}' not found in DataFrame columns.")
        else:
            # Try as column name
            if search_col in df.columns:
                col_idx = df.columns.get_loc(search_col)
            else:
                raise ValueError(f"Column '{search_col}' not found in DataFrame columns.")
    else:
        raise ValueError("search_col must be an int, str (column name), or Excel column letter.")

    # Get the column data up to the limit
    col_data = df.iloc[:limit, col_idx]

    for i, val in enumerate(col_data):
        if val == expected_label:
            excel_col = idx_to_excel_col(col_idx)
            excel_row = i + 1  # Excel rows are 1-based
            return f"{excel_col}{excel_row}"
    # return None if no matches found
    return None

def checker_valid_format_aft2024(filepath):
    """for a single file, runs a series of formatting checks. If pass, will return True, else returns {filename, reason}"""
    
    table = ["List Price",
             "GTN % excluding Launch yr one-time costs",
             "Launch Yr one-time cost (Listing fees, launch COOP)",
             "Launch Year ASP Excluding Launch yr one-time costs",
             "Yr 2 ongoing ASP Excluding Launch yr one-time costs",
             "COGS/unit",
             "Other COGS/",
             "One-time costs (Clearance, W/off)", 
             "GP % including One-Time launch yr costs",
             "GP % Yr 2 onwards",
             ]
    
    try: 
        df = pd.read_excel(
                f"{filepath}", #change your directory for file loading
                sheet_name="Project Summary",
                header=None
            )

        
        # Check for all items in table in column index 2
        for item in table:
            if not (df.iloc[:,2] == item).any():
                return {filepath: f'"{item}" not found in column index 2'}

    except Exception as e:
        return {filepath : "no sheet found"}

def extract_excel_table_to_df(filepath, sheet_name, start_cell, end_cell, header_row=False):
    """
    Extracts a range of values from an Excel sheet into a DataFrame.
    
    Parameters:
        filepath (str): Path to the Excel file.
        sheet_name (str|int): Sheet name or index.
        start_cell (str): Top-left cell (e.g., "C18").
        end_cell (str): Bottom-right cell (e.g., "I35").
        header_row (bool): If True, treat the first row of the extracted range as header.
    
    Returns:
        pd.DataFrame: Extracted range as a DataFrame.
    """
    # Read entire sheet without header so we keep raw layout
    df_raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None, engine="openpyxl")

    # Convert cell references (e.g., "C18") → row/col numbers
    from openpyxl.utils import coordinate_to_tuple
    
    start_row, start_col = coordinate_to_tuple(start_cell)
    end_row, end_col = coordinate_to_tuple(end_cell)
    
    # Pandas is 0-indexed; Excel is 1-indexed → adjust
    df_range = df_raw.iloc[start_row-1:end_row, start_col-1:end_col]

    if header_row:
        # Use the first row as header, then drop it from the data
        df_range.columns = df_range.iloc[0]
        df_range = df_range.iloc[1:].reset_index(drop=True)
    else:
        df_range = df_range.reset_index(drop=True)
    
    return df_range


### [IMPT] New 2024/25, net sales focused extractions

In [38]:
import pandas as pd
import re
import country_converter as coco
import openpyxl
import os

#helper
def extract_project_name(filepath):
    """
    Extracts the project name from a filename string.
    The project name is defined as the first item between the first pair of hyphens.
    If the extracted project name contains numbers, outputs a warning message.
    Example: "G1 P&L - Clear - CANZ & AMENA - MR - Inno (I-L).xlsm" -> "Clear"
    Now, always extracts from the last component if '/' is present.
    """
    # Always operate on the filename only (last component after '/')
    filename = filepath.split('/')[-1] if '/' in filepath else filepath
    matches = re.findall(r'-\s*([^-\n\r]+?)\s*-', filename) 
    if matches:
        project_name = matches[0].strip() #extracts the first item between the first pair of hyphens
        return project_name
    else:
        return None

#helper
def extract_year(filepath):
    """
    Extracts the year (e.g., 2023, 2024, 2025, etc.) from a filename or path.
    Looks for a 4-digit number starting with '20' in any component of the path.
    Returns the first such year found as a string, or None if not found.
    """
    # Split the path into components using common delimiters
    components = re.split(r'[_\-\s\.\\/]', filepath)
    for comp in components:
        match = re.search(r'20\d{2}', comp)
        if match:
            return match.group(0)
    return None

def extract_gate_number(filepath):
    """
    Extracts the gate number (e.g., 'G1', 'G2', etc.) from a filename or path.
    Looks for the first segment before a hyphen in the last path component,
    and returns it if it matches the 'G' followed by a number pattern.
    """
    # Get the last part of the path (after the last '/'), or the whole string if no '/'
    filename = os.path.basename(filepath)
    # Extract the first segment before the first hyphen
    match = re.match(r'^\s*([^-\n\r]+?)\s*-', filename)
    if match:
        item = match.group(1).strip()
        # Look for 'G' followed by a number (e.g., G1, G2, etc.)
        g_match = re.search(r'G\d+', item)
        if g_match:
            return g_match.group(0)
        else:
            return None
    else:
        return None

def extract_excel_table_to_df(filepath, sheet_name, start_cell, end_cell, header_row=False):
    """
    Helper function that extracts a range of values from an Excel sheet into a DataFrame.
    
    Parameters:
        filepath (str): Path to the Excel file.
        sheet_name (str|int): Sheet name or index.
        start_cell (str): Top-left cell (e.g., "C18").
        end_cell (str): Bottom-right cell (e.g., "I35").
        header_row (bool): If True, treat the first row of the extracted range as header.
    
    Returns:
        pd.DataFrame: Extracted range as a DataFrame.
    """
    # Read entire sheet without header so we keep raw layout
    df_raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None, engine="openpyxl")

    # Convert cell references (e.g., "C18") → row/col numbers
    from openpyxl.utils import coordinate_to_tuple
    
    start_row, start_col = coordinate_to_tuple(start_cell)
    end_row, end_col = coordinate_to_tuple(end_cell)
    
    # Pandas is 0-indexed; Excel is 1-indexed → adjust
    df_range = df_raw.iloc[start_row-1:end_row, start_col-1:end_col]

    if header_row:
        # Use the first row as header, then drop it from the data
        df_range.columns = df_range.iloc[0]
        df_range = df_range.iloc[1:].reset_index(drop=True)
    else:
        df_range = df_range.reset_index(drop=True)
    
    return df_range #dense df with sku name identifier

def clean_list_price_df(df):
    """
    Cleans / extracts out DataFrame by:
    - Finding the 'List Price' column
    - Dropping rows where 'List Price' contains 'input' (case-insensitive)
    - Dropping columns that are all NaN or all 0
    - Dropping the row immediately below the header row

    Args:
        df (pd.DataFrame): The input DataFrame

    Returns:
        pd.DataFrame: The cleaned DataFrame
    """
    # Find the "List Price" column
    list_price_col = df.columns[df.columns.str.contains("List Price", case=False, na=False)]
    if not list_price_col.empty:
        col_name = list_price_col[0]
        # Drop rows where "List Price" contains "input" (case-insensitive, as string)
        df = df[~df[col_name].astype(str).str.contains("input", case=False, na=False)]

    df = df.dropna(axis=1, how='all') # Drop columns that are all NaN or all 0 (after dropping rows)
    df = df.loc[:, ~((df == 0) | (df.isna())).all(axis=0)]

    df = df.drop(df.index[0]).reset_index(drop=True) # Drop the row immediately below the header row
    return df

def melt_list_price_df(df):
    """
    Melts the cleaned list price DataFrame to long format with Sku_Market, sku_name, Market, and list_price_AUD columns.

    Args:
        df (pd.DataFrame): The cleaned DataFrame with 'List Price' and market columns.

    Returns:
        pd.DataFrame: Melted DataFrame with columns ['Sku_Market', 'sku_name', 'Market', 'list_price_AUD'] and filtered for nonzero, non-NaN prices.
    """
    # Melt the dataframe
    df_melted = df.melt(
        id_vars=["List Price"],
        var_name="Market",
        value_name="list_price_AUD"
    )

    # Create the Sku_Market column
    df_melted["Sku_Market"] = df_melted["List Price"] + "_" + df_melted["Market"]
    df_melted["sku_name"] = df_melted["Sku_Market"].str.split("_").str[0]

    # Reorder columns
    df_final = df_melted[["Sku_Market", "sku_name", "Market", "list_price_AUD"]]
    # Drop rows that have "Value" == NaN or 0
    df_final = df_final[(df_final["list_price_AUD"].notna()) & (df_final["list_price_AUD"] != 0)]
    return df_final

def extract_append_pnl_feature(df, field_name, start_cell, end_cell, excel_path="2025_flat_t/G1 P&L - Wolverine - CANZ AMENA - GLOBAL - Inno (I-L).xlsm", sheet_name="Project Summary"):
    """
    Extracts a feature from a PNL sheet (table) Appends a column to df by extracting a table from Excel and looking up values by sku_name and Market.

    Args:
        df (pd.DataFrame): The dataframe to append to (must have 'sku_name' and 'Market' columns).
        field_name (str): The name of the new column to append.
        start_cell (str): Top-left cell of the Excel table to extract.
        end_cell (str): Bottom-right cell of the Excel table to extract.
        excel_path (str): Path to the Excel file.
        sheet_name (str): Sheet name in the Excel file.

    Returns:
        pd.DataFrame: The input df with the new column appended.
    """
    # Extract the relevant table from Excel
    df_feature = extract_excel_table_to_df(excel_path, sheet_name, start_cell, end_cell, header_row=True)
    # Optionally print for debugging
    print(df_feature)
    # print(df_feature.iloc[0,0])

    # Set the leftmost column as the row index
    df_feature.set_index(df_feature.columns[0], inplace=True)

    # Define lookup function
    def lookup_value(row):
        try:
            return df_feature.loc[row['sku_name'], row['Market']]
        except KeyError:
            return None

    # Append the new column
    df[field_name] = df.apply(lookup_value, axis=1)
    return df

def extract_project_summary(df, excel_path, sheet_name="Project Summary"):
    """
    Extracts forecast volume and net sales fields for 3 years from the Project Summary sheet
    and appends them to the input DataFrame.

    Args:
        df (pd.DataFrame): The dataframe to append to (must have 'sku_name' and 'Market' columns).
        excel_path (str): Path to the Excel file.
        sheet_name (str): Sheet name in the Excel file.

    Returns:
        pd.DataFrame: The input df with the new columns appended.
    """
    # This will work if excel_path always has at least two components separated by "/"
    filename = excel_path.split("/")[-1] #take the last component
    file_year = extract_year(excel_path)
    gate_no = extract_gate_number(excel_path)
    proj_name = extract_project_name(excel_path) #go above in extracting file

    # Gets the specific sku_name (based on combination of given name x market -- e.g Betadine Wound Healing Gel 10g_Australia)
    df = extract_excel_table_to_df(excel_path, "Project Summary", "C3","AI14", header_row = True) #Hard coded search within the list price table for non-zero values (indicates the sku was planned)
    df = clean_list_price_df(df) # Clean the extracted DataFrame
    df = melt_list_price_df(df) #get the base
    
    fields = [
        # (field_name, start_cell, end_cell)
        # WHITE TABS
        ("forecast_volume_y1", "BW2",  "DC15"),
        ("forecast_volume_y2", "BW17", "DC30"),
        ("forecast_volume_y3", "BW32", "DC45"),
        ("forecast_volume_y4", "BW47", "DC58"),
        ("forecast_volume_y5", "BW62", "DC73"), #Volumes TAB

        #LIST PRICE --> #Selling Price TAB

        ("GTN_perc excluding Launch yr one-time costs", "C16", "AI27"), #YELLOW TAB
        ("COGS/unit", "C69", "AI80"), #YELLOW TAB 
        ("Launch Yr one-time cost (Listing fees, launch COOP)", "C30", "AI41"), #YELLOW TAB, #COGS tab

        ("forecast_net_sales_y1", "HD2",  "IJ15"),
        ("forecast_net_sales_y2", "HD17", "IJ30"),
        ("forecast_net_sales_y3", "HD32", "IJ45"),
        ("forecast_net_sales_y4", "HD47", "IJ58"),
        ("forecast_net_sales_y5", "HD62", "IJ73"),

        ("launchyr_coop_listingfees_y1", "FV2", "HB15"),
        ("launchyr_coop_listingfees_y2", "FV17", "HB30"),
        ("launchyr_coop_listingfees_y3", "FV32", "HB45"),
        ("other_COGS_y1", "JT2", "KZ13"),
        ("other_COGS_y2", "JT17", "KZ28"),
        ("other_COGS_y3", "JT32", "KZ43"),
        ("other_COGS_y4", "JT47", "KZ58"),
        ("other_COGS_y5", "JT62", "KZ73"),

        ("one_time_costs_y1", "LC2", "MI13"),
        ("one_time_costs_y2", "LC17", "MI28"),
        ("one_time_costs_y3", "LC32", "MI43"),
        ("one_time_costs_y4", "LC47", "MI58"),
        ("one_time_costs_y5", "LC62", "MI73"),

        ("GP_y1", "ML2", "NP13"),
        ("GP_y2", "ML17", "NP28"),
        ("GP_y3", "ML32", "NP43"),
        ("GP_y4", "ML47", "NP58"),
        ("GP_y5", "ML62", "NP73"),    

        ("forecast_gross_sales_y1", "DF2",  "EK15"),
        ("forecast_gross_sales_y2", "DF17",  "EK30"),
        ("forecast_gross_sales_y3", "DF32",  "EK45"),
        ("forecast_gross_sales_y4", "DF47",  "EK58"),
        ("forecast_gross_sales_y5", "DF62",  "EK73"),
        

    ]
    df_final = df.copy()
    print(df_final)

    for field_name, start_cell, end_cell in fields:
        df_final = extract_append_pnl_feature( #function should 
            df_final,
            field_name=field_name,
            start_cell=start_cell,
            end_cell=end_cell,
            excel_path=excel_path,
            sheet_name=sheet_name
        )

    # Add additional columns filename, sheet_name, file_year, project_name, gate_number to the front of df_final
    df_final.insert(0, 'filename', filename)
    df_final.insert(1, 'sheet_name', sheet_name)
    df_final.insert(2, 'file_year', file_year)
    df_final.insert(3, 'project_name', proj_name)
    df_final.insert(4, 'gate_number', gate_no)

    if 'Sku_Market' in df_final.columns:
        df_final = df_final.rename(columns={'Sku_Market': 'sku_name'})

    return df_final

def preproc_master_df(df):
    """
    Simple aggregations on the concatenated master_df before return.
    Adds COGS_other_combined_y1..y5 = one_time_costs_y{n} + other_COGS_y{n} (missing inputs treated as 0).
    """
    out = df.copy()

    def _num_series(frame, col_name):
        if col_name not in frame.columns:
            return pd.Series(0.0, index=frame.index, dtype=float)
        return pd.to_numeric(frame[col_name], errors="coerce").fillna(0.0)

    for y in range(1, 6):
        ot = f"one_time_costs_y{y}"
        oc = f"other_COGS_y{y}"
        dest = f"COGS_other_combined_y{y}"
        out[dest] = _num_series(out, ot) + _num_series(out, oc)
    return out

def extract_all_proj_summary_aft2024(directory_path, apply_preproc=False):
    """
    Iterates through all files in the given directory, runs extract_project_summary() on each,
    and appends the resulting DataFrames to a master DataFrame as additional rows.
    Returns the concatenated master DataFrame.
    Includes a progress counter of all items.
    Also tracks and returns a dict of filenames that were not processed (due to errors or empty DataFrames),
    with the reason for failure: 'no sheet found', 'more than 10 skus', 'check irregular row arrangement - index out of bounds', or 'others'.
    This version is updated to suit the latest extract_project_summary signature and output.

    Args:
        directory_path: Root folder to walk for Excel files.
        apply_preproc: If True, run ``preproc_master_df`` on ``master_df`` immediately before returning.
    """
    import os
    import pandas as pd

    print(f"Starting extract_all_proj_summary_aft2024. Current working directory: {os.getcwd()}")
    master_df = None
    not_processed_files = {}

    excel_files = []
    for root, dirs, files in os.walk(directory_path):
        for filename in files:
            if filename.endswith(('.xlsx', '.xls', '.xlsm')) and not filename.startswith('~'):
                filepath = os.path.join(root, filename)
                rel_path = os.path.relpath(filepath, directory_path)
                excel_files.append(rel_path)
    
    total_files = len(excel_files)
    print(f"Found {total_files} Excel files to process.")

    for idx, filename in enumerate(excel_files, 1):
        file_path = os.path.join(directory_path, filename)
        print(f"[{idx}/{total_files}] Processing: {file_path}")

        try:
            # The latest extract_project_summary expects only excel_path and sheet_name
            # and does not return a fail_reason, so we need to handle errors via exceptions
            df = extract_project_summary(None, file_path)  # Pass None for df, as per new signature
            # If df is None or empty, consider as not processed
            if df is not None and not df.empty:
                if master_df is None:
                    master_df = df
                else:
                    master_df = pd.concat([master_df, df], ignore_index=True)
            else:
                not_processed_files[filename] = "empty dataframe"
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            # Try to detect if it's an index error
            err_str = str(e).lower()
            if "no sheet found" in err_str:
                reason = "no sheet found"
            elif "more than" in err_str and "skus" in err_str:
                reason = "more than 10 skus"
            elif "index" in err_str and "out of bounds" in err_str:
                reason = "check irregular row arrangement - index out of bounds"
            else:
                reason = "others"
            not_processed_files[filename] = reason
            continue

    print(f"Finished processing {total_files} files.")
    if not_processed_files:
        print(f"The following files were not processed (due to errors or empty data):")
        for fname, reason in not_processed_files.items():
            print(f" - {fname}: {reason}")
    else:
        print("All files processed successfully.")
    if apply_preproc and master_df is not None and not master_df.empty:
        master_df = preproc_master_df(master_df)
    # Optionally, return both master_df and not_processed_files
    return master_df, not_processed_files

def helper_desiredlaunchyr_finder(filepath, sheet_name): #Yet to implement
    
    """Looks across the range H1 to S20 to find "Desired Launch Year", extracts the value to the right of it.
        Tests the extracted value to ensure it is valid, returning the value as a float, else returns None"""

    try:
        df2 = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    except Exception as e:
        print(f"Error: Failed to load sheet '{sheet_name}': {e}")  # loading error
        return None
    
    # extract all values within grid  (H to S , row 1 to row 20) --> find the cell with value matching "desired launch year" after value.lower() applied
    table = helper_extract_excel_table(filepath, sheet_name, "G12", "K16") #sheet_name should be "Key Metrics", cell H14
    # Find the item that is "Exchange Rate" (case-insensitive, strip spaces)
    exch_val = None
    # Iterate through the DataFrame to find the cell with "exchange rate"
    for row_idx in range(df2.shape[0]):
        for col_idx in range(df2.shape[1]):
            cell_value = df2.iloc[row_idx, col_idx]
            if str(cell_value).strip().lower() == "desired launch year":
                # Return the value in the same row, 1 column after
                if col_idx + 1 < df2.shape[1]:
                    exch_val = df2.iloc[row_idx, col_idx + 1]
                    if exch_val is None or pd.isna(exch_val) or exch_val == 0 or str(exch_val).strip().lower() == "nan":
                        print("desired launch year extracted is 0, nan, NaN, None")
                        return None
                    # check if exch_val can be cast to a float (i.e., not a string that can't be converted)
                    try:
                        exch_val_float = float(exch_val)
                    except (ValueError, TypeError):
                        print("desired launch year extracted is not a valid float")
                        return None
                    exch_val = exch_val_float
                    return exch_val
                break
    return exch_val



In [ ]:
master_df, not_processed_files = extract_all_proj_summary_aft2024("../pnl-data/old_format", apply_preproc= True)

master_df.head()

master_df.to_csv("extractions.csv")

### [IMPT!] Extraction of project lvl details

In [ ]:
# Macro extraction: extract project level details from all files in a directory

import os

def extract_project_level_details_for_file(excel_path, filename):
    sheet_name = 'Project Summary'
    aggregations = ['CANZ', 'Asia Direct', 'Asia Total', 'Europe Total', 'ME total', 'AMENA', 'Total iNova']
    
    # 0) Extract "active markets" (not aggregations, not NaN or 0)
    try:
        active_df = extract_excel_table_to_df(
            excel_path,
            sheet_name=sheet_name,
            start_cell='C3',
            end_cell='AI14',
            header_row=True
        )
    except Exception as e:
        print(f"Skip {filename}: {e}")
        return None

    # Drop the row immediately below the header (index 0)
    active_df = active_df.drop(active_df.index[0])

    # Drop columns: aggregations, all NaN or all zeros
    cols_to_keep = []
    for col in active_df.columns:
        if col in aggregations:
            continue
        col_values = pd.to_numeric(active_df[col], errors='coerce')
        if col_values.notna().sum() == 0:
            continue
        if col_values.fillna(0).sum() == 0:
            continue
        cols_to_keep.append(col)
    active_df = active_df[cols_to_keep]

    active_markets = []
    for col in active_df.columns:
        col_values = pd.to_numeric(active_df[col], errors='coerce')
        if (col_values.fillna(0) != 0).any():
            active_markets.append(col)

    def extract_active_market_metric_pivot(metric_df, active_markets, value_name):
        metric_active = metric_df.loc[:, [
            col for col in metric_df.columns
            if col in active_markets or col == metric_df.columns[0]
        ]]
        id_var = metric_active.columns[0]
        melted = metric_active.melt(
            id_vars=id_var,
            var_name='Market',
            value_name=value_name,
        )
        pivoted_metric = melted.pivot(
            index='Market',
            columns=id_var,
            values=value_name,
        ).reset_index()
        pivoted_metric.columns = [
            f"{value_name}_{col}" if col != 'Market' else 'Market'
            for col in pivoted_metric.columns
        ]
        return pivoted_metric

    try:
        # 1) Extract "ANP_perc of Net Sales" table
        anp_perc_df = extract_excel_table_to_df(
            excel_path,
            sheet_name=sheet_name,
            start_cell='C134',
            end_cell='AI140',
            header_row=True
        )
        pivoted = extract_active_market_metric_pivot(
            anp_perc_df,
            active_markets=active_markets,
            value_name='ANP_perc of Net Sales',
        )

        # 2) Extract "Incr Opex" table
        incr_opex_df = extract_excel_table_to_df(
            excel_path,
            sheet_name=sheet_name,
            start_cell='C147',
            end_cell='AI152',
            header_row=True,
        )
        incr_pivoted = extract_active_market_metric_pivot(
            incr_opex_df,
            active_markets=active_markets,
            value_name='Incr Opex (Submissions, FTE etc.)',
        )

        pivoted_all = pivoted.merge(incr_pivoted, on='Market', how='left')

        # 3) Extract CAPEX table -- skip first 5 data rows, drop blank
        third_df = extract_excel_table_to_df(
            excel_path,
            sheet_name=sheet_name,
            start_cell='C134',
            end_cell='AI145',
            header_row=True,
        )
        third_df = third_df.iloc[5:].dropna(how='all').reset_index(drop=True)
        third_pivoted = extract_active_market_metric_pivot(
            third_df,
            active_markets=active_markets,
            value_name='CAPEX',
        )
        pivoted_all = pivoted_all.merge(third_pivoted, on='Market', how='left')

        # Add filename column for provenance
        pivoted_all['filename'] = filename

        return pivoted_all
    except Exception as e:
        print(f"Skip {filename}, extraction error: {e}")
        return None

def extract_all_project_level_details(directory):
    results = []
    files = [f for f in os.listdir(directory)
             if f.endswith('.xlsm') or f.endswith('.xlsx')]
    for filename in files:
        excel_path = os.path.join(directory, filename)
        print(f"Processing {filename} ...")
        try:
            result = extract_project_level_details_for_file(excel_path, filename)
            if result is not None and not result.empty:
                results.append(result)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    if results:
        combined = pd.concat(results, ignore_index=True)
        combined.to_csv('ext.csv', index=False)
        return combined
    else:
        print("No data extracted from any file.")
        return None

# Usage: Call this macro for the directory
project_directory = '../pnl-data/old_format'
all_proj_details = extract_all_project_level_details(project_directory)
display(all_proj_details)


Processing ~$G3 P&L - Dexterity Medium Term - AU NZ - CANZ - Inno (R&D).xlsm ...
Skip ~$G3 P&L - Dexterity Medium Term - AU NZ - CANZ - Inno (R&D).xlsm: File is not a zip file
Processing ~$ANZ Yogi - iGate 3 Scope Change.xlsm ...
Skip ~$ANZ Yogi - iGate 3 Scope Change.xlsm: File is not a zip file
Processing ~$G3 P&L - Lego PH - PH - O.xlsx ...
Skip ~$G3 P&L - Lego PH - PH - O.xlsx: File is not a zip file
Processing ~$G4 P&L - Aura TH - TH - O.xlsx ...
Skip ~$G4 P&L - Aura TH - TH - O.xlsx: File is not a zip file
Processing ANZ Yogi - iGate 3 Scope Change.xlsm ...


/opt/anaconda3/envs/inova/lib/python3.10/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/anaconda3/envs/inova/lib/python3.10/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/anaconda3/envs/inova/lib/python3.10/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/anaconda3/envs/inova/lib/python3.10/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


,Market,ANP_perc of Net Sales_nan,ANP_perc of Net Sales_Year 1,ANP_perc of Net Sales_Year 2,ANP_perc of Net Sales_Year 3,ANP_perc of Net Sales_Year 4,ANP_perc of Net Sales_Year 5,"Incr Opex (Submissions, FTE etc.)_Year 1","Incr Opex (Submissions, FTE etc.)_Year 2","Incr Opex (Submissions, FTE etc.)_Year 3","Incr Opex (Submissions, FTE etc.)_Year 4","Incr Opex (Submissions, FTE etc.)_Year 5",CAPEX_CAPEX $,filename
0,Australia,NaN,0.05,0.05,0.02,0.02,0.02,NaN,NaN,NaN,NaN,NaN,75727,ANZ Yogi - iGate 3 Scope Change.xlsm
1,NZ,NaN,0.05,0.05,0.02,0.02,0.02,NaN,NaN,NaN,NaN,NaN,122000,ANZ Yogi - iGate 3 Scope Change.xlsm


### Combining old/ new extracts 

In [20]:
df2024 = pd.read_csv("2024_volume_extracts.csv")
df2025 = pd.read_csv("2025_volume_extracts.csv")

df_combined = pd.concat([df2024, df2025], ignore_index=True)

# Remove 'Unnamed: 0' column if present
if 'Unnamed: 0' in df_combined.columns:
    df_combined = df_combined.drop(columns=['Unnamed: 0'])

# Optional: display or save the combined dataframe if needed
print(df_combined.head())
df_combined.to_csv("pnl2425_volume_extracts.csv")

                                       filename       sheet_name  file_year  \
0  G4 P&L - Rusty - AU NZ - CANZ - Exp (M).xlsm  Project Summary       2024   
1  G4 P&L - Rusty - AU NZ - CANZ - Exp (M).xlsm  Project Summary       2024   
2  G4 P&L - Rusty - AU NZ - CANZ - Exp (M).xlsm  Project Summary       2024   
3  G4 P&L - Rusty - AU NZ - CANZ - Exp (M).xlsm  Project Summary       2024   
4  G4 P&L - Rusty - AU NZ - CANZ - Exp (M).xlsm  Project Summary       2024   

  project_name gate_number                                           sku_name  \
0        Rusty          G4       Betadine Ready to Use Gargle 200mL_Australia   
1        Rusty          G4        Betadine Ready to Use Gargle 60mL_Australia   
2        Rusty          G4       Betadine Antiseptic Solution 500mL_Australia   
3        Rusty          G4  Betadine Antiseptic First Aid Cream 45g_Australia   
4        Rusty          G4              Betadine Ready to Use Gargle 200mL_NZ   

                                sku_na

### [Old] Listing Fees Extract (combining 2 files) for Matt Stenmark
- Instead, combine a) listing fees with b) 24/25 extractions

In [ ]:
df_2023 = pd.read_csv("csv-files/extracts_combined_20_23_III.csv")
df_2425 = pd.read_csv("csv-files/extracts_combined_24_25.csv")

df_25latest = pd.read_csv("listing_fees_25latest.csv")


columns_to_keep = [
    "filename", "file_year", "project_name", "gate_number", "sku_name", 
    "Listing Fees (LC,000)_launchquarter", "Listing Fees (LC,000)_y1", "Listing Fees (LC,000)_y2", "Listing Fees (LC,000)_y3", "Listing Fees (LC,000)_y4", "Listing Fees (LC,000)_y5",
    "List Price (LC)_launchquarter_AUDconverted", "List Price (LC)_y1_AUDconverted", "List Price (LC)_y2_AUDconverted", "List Price (LC)_y3_AUDconverted", "List Price (LC)_y4_AUDconverted", "List Price (LC)_y5_AUDconverted",
    "Gross Sales (LC,000)_launchquarter_AUDconverted", "Gross Sales (LC,000)_y1_AUDconverted", "Gross Sales (LC,000)_y2_AUDconverted", "Gross Sales (LC,000)_y3_AUDconverted", "Gross Sales (LC,000)_y4_AUDconverted", "Gross Sales (LC,000)_y5_AUDconverted",
    "Trade Spend Per (disc/bonus gds)_launchquarter_AUDconverted", "Trade Spend Per (disc/bonus gds)_y1_AUDconverted", "Trade Spend Per (disc/bonus gds)_y2_AUDconverted", "Trade Spend Per (disc/bonus gds)_y3_AUDconverted", "Trade Spend Per (disc/bonus gds)_y4_AUDconverted", "Trade Spend Per (disc/bonus gds)_y5_AUDconverted",
    "Trade Spend (LC,000)_launchquarter_AUDconverted", "Trade Spend (LC,000)_y1_AUDconverted", "Trade Spend (LC,000)_y2_AUDconverted", "Trade Spend (LC,000)_y3_AUDconverted", "Trade Spend (LC,000)_y4_AUDconverted", "Trade Spend (LC,000)_y5_AUDconverted",
    "Listing Fees (LC,000)_launchquarter_AUDconverted", "Listing Fees (LC,000)_y1_AUDconverted", "Listing Fees (LC,000)_y2_AUDconverted", "Listing Fees (LC,000)_y3_AUDconverted", "Listing Fees (LC,000)_y4_AUDconverted", "Listing Fees (LC,000)_y5_AUDconverted"
]

df_2023 = df_2023[columns_to_keep]

columns_to_keep_2425 = [
    "filename", "file_year", "project_name", "gate_number", "sku_name", "country_region", "list_price", "gtn_pct_excl_onetime",
    "launchyr_onetime_cost", "launchyr_asp_excl_onetime", "yr2_ongoing_asp_excl_onetime", "cogs_per_unit", "other_cogs_per_unit", "onetime_costs_clearance_woff"
]
df_2425 = df_2425[columns_to_keep_2425]

# Build the correct column order:
# Start with the five base fields
base_fields = ["filename", "file_year", "project_name", "gate_number", "sku_name"]

# Then all columns from df_2023 except the base fields
df_2023_extra = [col for col in columns_to_keep if col not in base_fields]

# Then all columns from df_2425 except the base fields and not in df_2023
df_2425_extra = [col for col in columns_to_keep_2425 if col not in base_fields and col not in df_2023_extra]

# Concatenate to get final column order
ordered_columns = base_fields + df_2023_extra + df_2425_extra

# Align both dataframes to the full ordered columns
df_2023_aligned = df_2023.reindex(columns=ordered_columns)
df_2425_aligned = df_2425.reindex(columns=ordered_columns)

# Concatenate the dataframes, stacking rows
df_combined = pd.concat([df_2023_aligned, df_2425_aligned], ignore_index=True)

# Optional: display or save the combined dataframe if needed
print(df_combined.head())
df_combined.to_csv("filetest.csv")


### LATEST GATE VERSION

# This will work even if the max gate_number differs for each project (including when the max is G1).
# For each project_name, it will retain only those rows with the highest gate_number (numerically, e.g., G1, G2, ..., Gx) for that project.

def gate_to_num(gate):
    if isinstance(gate, str) and gate.upper().startswith('G'):
        try:
            return int(gate[1:])
        except:
            return None
    return None

# Add a numeric column for gate_number (None if not present/invalid)
df_combined['gate_number_num'] = df_combined['gate_number'].apply(gate_to_num)

# Group by project_name and keep only rows with the max gate_number_num (regardless if G1, G2, ..., Gx)
df_combined = df_combined[df_combined.groupby('project_name')['gate_number_num'].transform('max') == df_combined['gate_number_num']]
df_combined.drop(columns=['gate_number_num'], inplace=True)
df_combined.to_csv("filetest2.csv")


### Parsing Numeric column for gate for latest 2025 extractions (G4 -> 4)

In [78]:
df_25latest = pd.read_csv("final_listing_fees_combined_24_25.csv")

def gate_to_num(gate):
    if isinstance(gate, str) and gate.upper().startswith('G'):
        try:
            return int(gate[1:])
        except:
            return None
    return None

# Add a numeric column for gate_number (None if not present/invalid)
df_25latest['gate_number_num'] = df_25latest['gate_number'].apply(gate_to_num)

# Group by project_name and keep only rows with the max gate_number_num (regardless if G1, G2, ..., Gx)
df_25latest = df_25latest[df_25latest.groupby('project_name')['gate_number_num'].transform('max') == df_25latest['gate_number_num']]
df_25latest.drop(columns=['gate_number_num'], inplace=True)
df_25latest.to_csv("final_listing_fees_combined_24_25_latest_gate.csv")


#### Combined Listing Fees with 24_25 extracts
- re run for 24 / 25 directories, then append to listing_fees_25latest_lastgate.csv

In [ ]:
master_df_2024, not_processed_files_24 = extract_all_proj_summary_aft2024("2024_flat")
master_df_2024.to_csv("pnl_listing_fees_24.csv")

master_df_2025, not_processed_files_25 = extract_all_proj_summary_aft2024("2025_flat")
master_df_2025.to_csv("pnl_listing_fees_25.csv")

# # Append all rows from master_df_2024, master_df_2025, and df_25latest into a single DataFrame
# combined_listing_fees = pd.concat([master_df_2024, master_df_2025], ignore_index=True)
# combined_listing_fees.to_csv("listing_fees_combined_24_25.csv", index=False)

In [77]:
df_latest= pd.read_csv("listing_fees_25latest_lastgate.csv")
df2425 = pd.read_csv("listing_fees_combined_24_25.csv")
combined_listing_fees = pd.concat([df_latest, df2425], ignore_index=True)

combined_listing_fees.to_csv("final_listing_fees_combined_24_25.csv", index=False)

### Notes: 2023 PNL Extraction Algo Pseudo Code

- checker
    - check "Sales Forecast(LC)" sheet exists -- else "no sheet found"
    - "Description" exists in Col D -- else "sheet format different"
        - iteratively read rows in col D, if exceed 40, break
    
    - check no. valid tables, compare w no.skus -- else "sku and table value mismatch"
        - iteratively read rows in col D, adding to a list sku_names commencing from when the first value 'Description' is encountered. Break when '20xx' is encountered, or if exceed 40 rows, break
         
        - check for non-zero columns in Unit Cost/ List Price row  = 1 table
        - iteratively read rows in col C, when "List Price (LC)" is encountered, for the same row, read 1,2,3,4 cells to the right (iterate columns for the same row). If at least 1 cell is non-zero (or NaN), increment n_tables by 1. Break when "Revenue & Margin Impact - Cannibalized or Umbrella effect on existing brand" encountered, or if exceed 1000 rows (output: '1000 rows exceeded').

        - if len(sku_names) > 15, output: "more than 15 skus"
        - if len(sku_names) = 0 output: "no SKUs found" 
        - compare n_tables with len(sku_names). If no equal, output: "unequal sku names and tables, count: sku: x, tables: y"
        
    - check exchange rate value exists -- else "no exchange rate found" (Valuation (AUD), M14)
    return boolean pass/fail check

- reader 
    - load sales forecast sheet
        - read gate name (extract_year) -- run on directory path
        - read project name (extract_project_name) -- run on filenames
        - read gate name (extract_gate_number) -- run on filenames

        #sheet name
        #sku_name
        #metric
        #value
    
    - read values
        -  iteratively read rows in col D, starting from after the first value 'Description' is encountered, for subsequent values, read 1,2,3 cells to the right (iterate columns for the same row). Inclusive of the current cell value, concatenate all 4 cells together with a space separating the contents of each cell. Add the resulting string to sku_names
        Break when '20xx' is encountered, or if exceed 40 rows, break.

        - try to load the excel file
        -  iteratively read rows in col D, starting from after the first value 'Description' is encountered, for subsequent values, read 1,2,3 cells to the right (iterate columns for the same row). Inclusive of the current cell value, concatenate all 4 cells together with a space separating the contents of each cell. Add the resulting string to sku_names
                Break when '20xx' is encountered, or if exceed 40 rows, break.

        output_dict = {}
        - Maintain a df_counter. 
        - iteratively read rows in col D, when the first 2024 is encountered, look up the row index (cell_start= 'C'+row_start)
            - row_end = row_start + 18; cell_end= 'I'+ row_end
            - run extract_table_persku(filepath, sheet_name, cell_start, cell end) #returns dict
            - save extract_table_persku output as a value in output_dict e.g {sku_names[df_counter]: value}
            
            increment df_counter
            break when df_counter = len(sku_names)
        
        - return output_dict (dictionary with key-value pairs for each sku in a given excel file)
        

        - Save sku values for each sku as a df (to be converted to dict)
            - Maintain a df_counter. iteratively read rows in col D, when "20xx" is encountered, look up the row index (row_start). 
            - Create a df of name df_{df_counter} from row_start, col C to (row_start + 18), Col I, then increment df_counter
            - break when df_counter = len(sku_names)


        - divide to convert local currency to AUD
            - 'Sales volumne pull through (,000)', 'Unit Cost (LC)', 'List Price (LC)', 'Gross Sales (LC,000)', 'Trade Spend Per (disc/bonus gds)', 'Trade Spend (LC,000)', 'Listing Fees (LC,000), "Avg Net Selling Price (LC)", "Pipeline fill (LC,000)", "Net Sales (LC,000)", "Gross Margin (LC,000)"

    
[ ] conversion of year / metric
[ ] preservation of AUD

supposed dictionary format:
